In [1]:
# %pip install transformers evaluate torch peft
# %pip install accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [19]:
# %pip install torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 6.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
# import os
# print(os.getcwd())


/content


# Set up

In [2]:
import torch
from transformers import RobertaModel, RobertaTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model
import evaluate
from datasets import Dataset


In [3]:
import pandas as pd

In [4]:
frozen_train = pd.read_csv('train_dataset.csv')

In [5]:
frozen_val = pd.read_csv('val_dataset.csv')

In [6]:
frozen_test = pd.read_csv('test_dataset.csv')

# Load model and basic preprocessing

In [7]:
# load model
peft_model_name = 'roberta-base-peft'
modified_base = 'roberta-base-modified'
base_model = 'roberta-base'

In [8]:
# processing
tokenizer = RobertaTokenizer.from_pretrained(base_model)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [21]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

In [22]:
def to_tokenized(df):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(preprocess_function, batched=True, remove_columns=['text'])

In [23]:
tokenized_train_dataset = to_tokenized(frozen_train)
tokenized_val_dataset = to_tokenized(frozen_val)
tokenized_test_dataset = to_tokenized(frozen_test)

Map:   0%|          | 0/65401 [00:00<?, ? examples/s]

Map:   0%|          | 0/8218 [00:00<?, ? examples/s]

Map:   0%|          | 0/8151 [00:00<?, ? examples/s]

In [12]:
id2label = {0: 'human', 1: 'LLM-generated'}


In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [24]:
import numpy as np

In [25]:
tokenized_train_dataset.column_names

['label', 'input_ids', 'attention_mask']

In [26]:
# lens = [len(x) for x in tokenized_train_dataset["input_ids"]]
# print(np.percentile(lens, [50, 90, 99]), max(lens))

# Training

    group_by_length=True

In [29]:
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='steps',
    eval_steps=200,
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    logging_steps=50,
)

In [30]:
def get_trainer(model):
      return  Trainer(
          model=model,
          args=training_args,
          train_dataset=tokenized_train_dataset,
          eval_dataset=tokenized_val_dataset,
          data_collator=data_collator,
      )

In [31]:
peft_config = LoraConfig(task_type="SEQ_CLS",
                         inference_mode=False,
                         r=8,
                         lora_alpha=16,
                         lora_dropout=.1)


In [33]:
model = AutoModelForSequenceClassification.from_pretrained(base_model, id2label=id2label)

peft_model = get_peft_model(model, peft_config)

print('PEFT Model')
peft_model.print_trainable_parameters()

peft_lora_finetuning_trainer = get_trainer(peft_model)

peft_lora_finetuning_trainer.train()
peft_lora_finetuning_trainer.evaluate()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


PEFT Model
trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


Step,Training Loss,Validation Loss
200,0.112487,0.187074
400,0.057080,0.110626
600,0.045560,0.068231
800,0.063132,0.153513
1000,0.016948,0.045991
1200,0.014421,0.035540
1400,0.129275,0.111252
1600,0.020533,0.119145
1800,0.071938,0.344821
2000,0.075876,0.243757


Training Loss,Validation Loss,Step
0.000050,0.074475,4088


{'eval_loss': 0.0744745060801506}

In [34]:
tokenizer.save_pretrained(modified_base)
peft_model.save_pretrained(peft_model_name)

In [36]:
from peft import AutoPeftModelForSequenceClassification

In [37]:
inference_model = AutoPeftModelForSequenceClassification.from_pretrained(peft_model_name, id2label=id2label)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
from torch.utils.data import DataLoader
from tqdm import tqdm

In [38]:
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def evaluate_model(inference_model, dataset):

    eval_dataloader = DataLoader(dataset.rename_column("label", "labels"), batch_size=8, collate_fn=data_collator)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    inference_model.to(device)
    inference_model.eval()
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch.to(device)
        with torch.no_grad():
            outputs = inference_model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        predictions, references = predictions, batch["labels"]
        accuracy.add_batch(
            predictions=predictions,
            references=references,
        )
        f1.add_batch(
            predictions=predictions,
            references=references,
        )
    acc_results = accuracy.compute()
    f1_results = f1.compute()
    print("accuracy: ", acc_results, " f1_results: ", f1_results)

In [39]:
evaluate_model(inference_model, tokenized_test_dataset)


100%|██████████| 1019/1019 [02:03<00:00,  8.28it/s]

accuracy:  {'accuracy': 0.9915347810084653}  f1_results:  {'f1': 0.9869737587313574}
